# Task 29: Assignment


In [1]:
from datasets import load_dataset

ds = load_dataset("dair-ai/emotion")
print(ds)
print(ds['train'][0])
print(ds['train'].features)

README.md:   0%|          | 0.00/9.05k [00:00<?, ?B/s]

split/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.03MB            

split/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

split/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  127kB            

split/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

split/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/16000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})
{'text': 'i didnt feel humiliated', 'label': 0}
{'text': Value('string'), 'label': ClassLabel(names=['sadness', 'joy', 'love', 'anger', 'fear', 'surprise'])}


In [2]:
import joblib
import pandas as pd
from datasets import load_dataset
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

Q1. (Data Preparation)

Load the cleaned Emotions dataset (or the original train.txt after cleaning).

Separate the features (text) into X and target (emotion) into y.

Perform train-test split with test_size=0.20 and random_state=42.

Print the shape of X_train, X_test, y_train, and y_test.


In [3]:
# Load the dataset from Hugging Face
ds = load_dataset("dair-ai/emotion")

# Convert the Hugging Face dataset split into a Pandas DataFrame
df = pd.DataFrame(ds["train"])


label_names = ds["train"].features["label"].names
df["emotion"] = df["label"].map(lambda x: label_names[x])

# Separate features (text) and target (emotion)
X = df["text"]
y = df["emotion"]

# Train-test split with test_size=0.20 and random_state=42
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape: ", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape: ", y_test.shape)

X_train shape: (12800,)
X_test shape:  (3200,)
y_train shape: (12800,)
y_test shape:  (3200,)


Q2. (Bag of Words – Basic)

Apply CountVectorizer (Bag of Words) on the training data:

Fit and transform X_train.

Transform X_test.

Print the shape of the resulting matrices.

Display the first 20 feature names (vocabulary words).


In [4]:
cv_basic = CountVectorizer()
X_train_bow = cv_basic.fit_transform(X_train)
X_test_bow = cv_basic.transform(X_test)

print("X_train_bow matrix shape:", X_train_bow.shape)
print("X_test_bow matrix shape: ", X_test_bow.shape)

feature_names = cv_basic.get_feature_names_out()
print("First 20 feature names (vocabulary):", list(feature_names[:20]))

X_train_bow matrix shape: (12800, 13501)
X_test_bow matrix shape:  (3200, 13501)
First 20 feature names (vocabulary): ['aa', 'aaaaand', 'aaaand', 'aac', 'aahhh', 'aaron', 'ab', 'abandon', 'abandoned', 'abandoning', 'abandonment', 'abated', 'abbigail', 'abc', 'abdomen', 'abdominal', 'abducted', 'abhorrent', 'abide', 'abilities']


Q3. (Bag of Words + MultinomialNB)

Train a Multinomial Naive Bayes model using the Bag of Words features:

Fit the model on X_train_bow and y_train.

Make predictions on the test set.

Calculate and print the accuracy score.

In [5]:
nb_bow = MultinomialNB()
nb_bow.fit(X_train_bow, y_train)

y_pred_bow = nb_bow.predict(X_test_bow)
acc_bow = accuracy_score(y_test, y_pred_bow)
print(f"BoW (Unigram) + MultinomialNB Accuracy: {acc_bow * 100:.2f}%")

BoW (Unigram) + MultinomialNB Accuracy: 73.91%


Q4. (Understanding Vocabulary)

Using the fitted CountVectorizer from Q2:

Print the total size of the vocabulary.

Show any 15 words from the vocabulary.

Convert one sample document from the training set into its BoW vector and
display it.


In [6]:
vocab_size = len(cv_basic.vocabulary_)
print("Total Vocabulary Size:", vocab_size)

# Display 15 sample words from vocabulary
sample_words = list(cv_basic.vocabulary_.keys())[:15]
print("Sample 15 words from vocabulary:", sample_words)

# Convert one sample document from X_train into its BoW vector
sample_text = [X_train.iloc[0]]
sample_vec = cv_basic.transform(sample_text)
print(f"Sample Document: '{sample_text[0]}'")
print("Non-zero term indices in BoW vector:", sample_vec.nonzero()[1])

Total Vocabulary Size: 13501
Sample 15 words from vocabulary: ['refers', 'of', 'course', 'though', 'cant', 'help', 'feeling', 'somehow', 'ironically', 'in', 'retrospect', 'to', 'loudons', 'son', 'with']
Sample Document: 'i refers of course though i cant help feeling somehow ironically in retrospect to loudons son with kate mcgarrigle the rather talented himself rufus wainwright'
Non-zero term indices in BoW vector: [ 1701  2622  4419  5504  5571  5931  6249  6514  7059  7338  8173  9521
  9661  9928 10127 10995 11007 11751 11938 12005 12112 12951 13260]


Q5. (N-grams with Bag of Words)

Create a new CountVectorizer that uses bigrams (ngram_range=(1, 2)).

Fit and transform the training data.

Transform the test data.

Print the shape of the new feature matrix.

Display some of the bigram features.


In [7]:
cv_ngram = CountVectorizer(ngram_range=(1, 2))
X_train_ngram = cv_ngram.fit_transform(X_train)
X_test_ngram = cv_ngram.transform(X_test)

print("X_train_ngram matrix shape:", X_train_ngram.shape)
print("X_test_ngram matrix shape: ", X_test_ngram.shape)

# Display sample bigram features
ngram_features = cv_ngram.get_feature_names_out()
bigrams = [f for f in ngram_features if " " in f][:15]
print("Sample 15 Bigram Features:", bigrams)

X_train_ngram matrix shape: (12800, 106150)
X_test_ngram matrix shape:  (3200, 106150)
Sample 15 Bigram Features: ['aa full', 'aa meeting', 'aaaaand tis', 'aaaand after', 'aac or', 'aahhh work', 'aaron has', 'abandon it', 'abandon me', 'abandon the', 'abandoned ask', 'abandoned believe', 'abandoned by', 'abandoning him', 'abandonment has']


Q6. (N-grams + Model Training)

Train a MultinomialNB model using the bigram features created in Q5.

Calculate and print the accuracy on the test set.

Compare this accuracy with the unigram (basic BoW) accuracy from Q3.


In [8]:
nb_ngram = MultinomialNB()
nb_ngram.fit(X_train_ngram, y_train)

y_pred_ngram = nb_ngram.predict(X_test_ngram)
acc_ngram = accuracy_score(y_test, y_pred_ngram)
print(f"N-grams (1, 2) + MultinomialNB Accuracy: {acc_ngram * 100:.2f}%")

N-grams (1, 2) + MultinomialNB Accuracy: 64.97%


Q7. (TF-IDF Vectorization)

Apply TfidfVectorizer on the same train-test split:

Fit and transform X_train.

Transform X_test.

Print the shape of the TF-IDF matrices.

Display the first 15 feature names.


In [9]:
tfidf = TfidfVectorizer()
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("X_train_tfidf matrix shape:", X_train_tfidf.shape)
print("X_test_tfidf matrix shape: ", X_test_tfidf.shape)

tfidf_features = tfidf.get_feature_names_out()
print("First 15 TF-IDF features:", list(tfidf_features[:15]))

X_train_tfidf matrix shape: (12800, 13501)
X_test_tfidf matrix shape:  (3200, 13501)
First 15 TF-IDF features: ['aa', 'aaaaand', 'aaaand', 'aac', 'aahhh', 'aaron', 'ab', 'abandon', 'abandoned', 'abandoning', 'abandonment', 'abated', 'abbigail', 'abc', 'abdomen']


Q8. (TF-IDF + MultinomialNB)

Train a Multinomial Naive Bayes model using the TF-IDF features:

Fit the model.

Make predictions on the test set.

Calculate and print the accuracy score.


In [10]:
nb_tfidf = MultinomialNB()
nb_tfidf.fit(X_train_tfidf, y_train)

y_pred_tfidf = nb_tfidf.predict(X_test_tfidf)
acc_tfidf = accuracy_score(y_test, y_pred_tfidf)
print(f"TF-IDF + MultinomialNB Accuracy: {acc_tfidf * 100:.2f}%")

TF-IDF + MultinomialNB Accuracy: 61.75%


Q9. (Comparison of Vectorizers)

Create a comparison table of the three approaches:

Bag of Words (Unigrams)

Bag of Words (Unigrams + Bigrams)

TF-IDF

Show the accuracy of each.

Write a short observation about which method
performed best and why.


In [11]:
comparison_df = pd.DataFrame(
    {
        "Vectorization Method": [
            "Bag of Words (Unigrams)",
            "Bag of Words (Unigrams + Bigrams)",
            "TF-IDF",
        ],
        "Accuracy": [
            f"{acc_bow * 100:.2f}%",
            f"{acc_ngram * 100:.2f}%",
            f"{acc_tfidf * 100:.2f}%",
        ],
    }
)
print(comparison_df.to_string(index=False))

             Vectorization Method Accuracy
          Bag of Words (Unigrams)   73.91%
Bag of Words (Unigrams + Bigrams)   64.97%
                           TF-IDF   61.75%


In [12]:
print("\nObservation:")
print(
    "- Bag of Words (Unigrams + Bigrams) captures phrase context (e.g., 'not happy'), "
    "which increases vocabulary size but often improves accuracy for sentiment analysis.\n"
    "- TF-IDF weights down common words that appear across all documents, giving more "
    "importance to rare, domain-specific emotion words."
)


Observation:
- Bag of Words (Unigrams + Bigrams) captures phrase context (e.g., 'not happy'), which increases vocabulary size but often improves accuracy for sentiment analysis.
- TF-IDF weights down common words that appear across all documents, giving more importance to rare, domain-specific emotion words.


Q10. (Mini Project – Complete Vectorization Pipeline)

Using the Emotions dataset:
1. Load and clean the text (if not already cleaned).
2. Perform train-test split.
3. Apply both CountVectorizer and TfidfVectorizer.
4. Train MultinomialNB on both.
5. Compare their accuracy.
6. Save the best vectorizer and the best model using joblib.

In [13]:
pipelines = {
    "CountVectorizer (Unigram)": (acc_bow, cv_basic, nb_bow),
    "CountVectorizer (Unigram+Bigram)": (acc_ngram, cv_ngram, nb_ngram),
    "TfidfVectorizer": (acc_tfidf, tfidf, nb_tfidf),
}

best_name = max(pipelines, key=lambda k: pipelines[k][0])
best_acc, best_vectorizer, best_model = pipelines[best_name]

# Save best vectorizer and model using joblib
joblib.dump(best_vectorizer, "best_vectorizer.pkl")
joblib.dump(best_model, "best_emotion_model.pkl")

print(f"Best Method Selected: {best_name}")
print(f"Best Accuracy Achieved: {best_acc * 100:.2f}%")
print("Saved 'best_vectorizer.pkl' and 'best_emotion_model.pkl' successfully!")

Best Method Selected: CountVectorizer (Unigram)
Best Accuracy Achieved: 73.91%
Saved 'best_vectorizer.pkl' and 'best_emotion_model.pkl' successfully!
